In [2]:
import jax
from flax import nnx
import jax.numpy as jnp
import optax

In [3]:
# model simple vector field network v_theta(x, t)
class HiddenNet(nnx.Module):
    def __init__(self, dim, *, rngs: nnx.Rngs) -> None:
        self.linear = nnx.Linear(dim, dim, rngs=rngs)

    def __call__(self, x):
        return nnx.softplus(self.linear(x))

class VectorFieldNetwork(nnx.Module):

    def __init__(self, input_dim=2, hidden_dim=64, num_hidden_layers=5, *, rngs: nnx.Rngs) -> None:
        self.linear1 = nnx.Linear(input_dim + 1, hidden_dim, rngs=rngs)

        @nnx.split_rngs(splits=num_hidden_layers)
        @nnx.vmap(in_axes=0, out_axes=0)
        def create_hidden_layer(rngs):
            return HiddenNet(hidden_dim, rngs=rngs)

        self.hidden_layers = create_hidden_layer(rngs)
        self.linear3 = nnx.Linear(hidden_dim, input_dim, rngs=rngs)

    def __call__(self, x, t):
        # x [batch_size, input_dim]
        # t [batch_size, 1]
        inputs = jnp.concatenate([x, t], axis=-1) # [batch_size, input_dim + 1]
        out1 = nnx.softplus(self.linear1(inputs))

        @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        def apply_hidden_layers(x, layer):
            x = layer(x)
            return x

        h = apply_hidden_layers(out1, self.hidden_layers)
        return self.linear3(h) # [batch_size, input_dim]

In [4]:
# 2. Helper to generate toy 2D target data (a circle)
def sample_target_data(batch_size, rng_key):
    k1, k2 = jax.random.split(rng_key)
    theta = jax.random.uniform(k1, (batch_size, 1)) * 2 * jnp.pi
    r = 2.0 + jax.random.uniform(k2, (batch_size, 1)) * 0.1
    x1 = jnp.concatenate([r * jnp.cos(theta), r * jnp.sin(theta)], axis=-1)  # [batch_size, 2]
    return x1

print(sample_target_data(5, jax.random.key(0)).shape)


Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/athapar/personal/ai-bio-projects/.venv/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 497, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/athapar/personal/ai-bio-projects/.venv/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 348, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/athapar/personal/ai-bio-projects/.venv/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 274, in _check_cuda_versions
    for d in range(cuda_versions.cuda_device_count())
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:126: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE


(5, 2)


In [ ]:
@nnx.jit
def train_step(model, optimizer, xt, t, u_t):
    def loss_fn(model):
        v_pred = model(xt, t)  # [batch_size, 2]
        return jnp.mean((v_pred - u_t) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss



def train_flow_matching(epochs=1000, batch_size=128, learning_rate=1e-3):
    model = VectorFieldNetwork(input_dim=2, hidden_dim=128, rngs=nnx.Rngs(0))
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)

    print("Starting training flow matching...")

    for epoch in range(epochs):
        epoch_key = jax.random.key(epoch)
        k1, k2, k3 = jax.random.split(epoch_key, num=3)

        # sample endpoint data: x1 ~ target distribution, x0 ~ simple prior (e.g., Gaussian)
        x1 = sample_target_data(batch_size, k1)
        x0 = jax.random.normal(k2, (batch_size, 2))  # [batch_size, 2]

        # sample random time t ~ Uniform(0, 1)
        t = jax.random.uniform(k3, (batch_size, 1)) # [batch_size, 1]

        # Construct the conditional path (Linear interpolation / Optimal Transport)
        xt = (1 - t) * x0 + t * x1  # [batch_size, 2]

        # Target velocity field u_t(x | x_0, x_1) = x_1 - x_0
        u_t = x1 - x0  # [batch_size, 2]

        loss = train_step(model, optimizer, xt, t, u_t)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    return model


In [6]:
# Sampling / Inference Function (Solving the ODE via Euler Integration)
@jax.jit(static_argnames=["num_samples", "num_steps"])
def sample_from_model(model, num_samples=1000, num_steps=50):
    model.eval()
    key = jax.random.key(42)
    # sample initial pure noise from simple prior x_0 ~ N(0, I)
    xt = jax.random.normal(key, (num_samples, 2))
    dt = 1.0 / num_steps

    def euler_step(i, carry):
        xt, model = carry
        t_val = i * dt
        t = jnp.full((num_samples, 1), t_val) # [num_samples, 1]

        vt = model(xt, t)  # [num_samples, 2]
        xt = xt + vt * dt
        return (xt , model)

    xt, _ = jax.lax.fori_loop(jnp.int32(0), num_steps,euler_step , (xt, model))

    return xt


In [7]:
trained_model = train_flow_matching(epochs=1001, batch_size=256, learning_rate=1e-3)


Starting training flow matching...
Epoch 0, Loss: 3.8360
Epoch 100, Loss: 2.8764
Epoch 200, Loss: 2.2474
Epoch 300, Loss: 2.1587
Epoch 400, Loss: 1.8608
Epoch 500, Loss: 2.2304
Epoch 600, Loss: 2.1449
Epoch 700, Loss: 2.0798
Epoch 800, Loss: 1.6796
Epoch 900, Loss: 1.7540
Epoch 1000, Loss: 2.1063


In [8]:

# Sample from the trained model
generated_samples = sample_from_model(trained_model, num_samples=5, num_steps=50)
print("\nGenerated 2D coordinates on the learned circle:")
print(generated_samples)


Generated 2D coordinates on the learned circle:
[[ 0.26678634  1.9040599 ]
 [ 1.839187    0.46235162]
 [-0.8044956   1.7033975 ]
 [-1.8059622   1.0032756 ]
 [ 1.330911    1.4711717 ]]


In [14]:
# Mean flows - https://arxiv.org/pdf/2412.06264 . get average speed and allow for 1 step inference.
class HiddenFlow(nnx.Module):
    def __init__(self, dim, *, rngs: nnx.Rngs) -> None:
        self.linear = nnx.Linear(dim, dim, rngs=rngs)

    def __call__(self, x):
        return nnx.silu(self.linear(x))

class MeanFlowNetwork(nnx.Module):

    def __init__(self, input_dim=2, hidden_dim=64, num_hidden_layers=5, *, rngs: nnx.Rngs) -> None:
        # Takes x, r, and t. Dimension = input_dim + 2
        self.linear1 = nnx.Linear(input_dim + 2, hidden_dim, rngs=rngs)
        self.hidden = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs)

        # @nnx.split_rngs(splits=num_hidden_layers)
        # @nnx.vmap(in_axes=0, out_axes=0)
        # def create_hidden_layer(rngs):
        #     return HiddenFlow(hidden_dim, rngs=rngs)

        # self.hidden_layers = create_hidden_layer(rngs)
        self.hidden_layers = nnx.List([HiddenFlow(hidden_dim, rngs=rngs) for _ in range(num_hidden_layers)])
        self.linear2 = nnx.Linear(hidden_dim, input_dim, rngs=rngs)

    def __call__(self, x, r, t):
        # x [batch_size, input_dim]
        # r [batch_size, 1]
        # t [batch_size, 1]
        inputs = jnp.concatenate([x, r, t], axis=-1) # [batch_size, input_dim + 2]
        out = nnx.silu(self.linear1(inputs))

        for layer in self.hidden_layers:
            out = layer(out)

        # @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        # def apply_hidden_layers(x, layer):
        #     x = layer(x)
        #     return x

        # h = apply_hidden_layers(out1, self.hidden_layers)

        return self.linear2(out) # [batch_size, input_dim]

In [ ]:
@nnx.jit
def train_step_mean_flow(model, optimizer, xt, r, t, u):
    def loss_fn(model):
        # Tangent vectors (the direction of changes with respect to dx/dt, dr/dt, dt/dt)
        # dx/dt along the path is exactly the velocity vector 'u'.
        # dr/dt = 0 (holding the start interval steady)
        # dt/dt = 1 (advancing the target clock)
        tangent_x = u
        tangent_r = jnp.zeros_like(r)
        tangent_t = jnp.ones_like(t)

        # Compute the network's prediction (v_pred) and its derivative along the path (dvdt)
        v_pred, dvdt = jax.jvp(lambda x, r_in, t_in: model(x, r_in, t_in), (xt, r, t),(tangent_x, tangent_r, tangent_t))

        # 4. The MeanFlow Identity targets:
        # The average velocity target relies on the current predictions and their derivatives
        u_target = u - (t - r) * dvdt

        # 5. Apply a stop gradient to the target side (just like in Reinforcement Learning or EMA targets)
        return jnp.mean((v_pred - jax.lax.stop_gradient(u_target)) ** 2)


    loss, grads = jax.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

def train_mean_flow_matching(epochs=1000, batch_size=128, learning_rate=1e-3):
    model = MeanFlowNetwork(input_dim=2, hidden_dim=128, rngs=nnx.Rngs(0))
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)

    print("Starting training mean flow matching...")

    for epoch in range(epochs):
        epoch_key = jax.random.key(epoch)
        k1, k2, k3, k4 = jax.random.split(epoch_key, num=4)

        # sample endpoint data: x1 ~ target distribution, x0 ~ simple prior (e.g., Gaussian)
        x1 = sample_target_data(batch_size, k1)
        x0 = jax.random.normal(k2, (batch_size, 2))

        # sample 2 2 time points r,t such that 0 <= r < t <= 1
        r = jax.random.uniform(k3, (batch_size, 1))
        t = r + jax.random.uniform(k4, (batch_size, 1)) * (1.0 - r)

        # Construct the conditional path (Linear interpolation / Optimal Transport)
        xt = (1 - t) * x0 + t * x1
        u = x1 - x0  # original instantaneous velocity field

        loss = train_step_mean_flow(model, optimizer, xt, r, t, u)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")
    return model


In [20]:
trained_mean_flow_model = train_mean_flow_matching(epochs=1001, batch_size=256, learning_rate=1e-3)

Starting training mean flow matching...
x shape: (256, 2), r shape: (256, 1), t shape: (256, 1)
Epoch 0, Loss: 3.0794
Epoch 100, Loss: 2.3068
Epoch 200, Loss: 2.7383
Epoch 300, Loss: 3.1631
Epoch 400, Loss: 3.7538
Epoch 500, Loss: 3.7370
Epoch 600, Loss: 3.6407
Epoch 700, Loss: 3.1606
Epoch 800, Loss: 2.9221
Epoch 900, Loss: 3.1755
Epoch 1000, Loss: 5.0493


In [21]:
def mean_flow_sample(model, num_samples):
    model.eval()
    key = jax.random.key(42)
    # sample initial pure noise from simple prior x_0 ~ N(0, I)
    x0 = jax.random.normal(key, (num_samples, 2))

    # define absoulte interval boundaries
    r = jnp.zeros((num_samples, 1))
    t = jnp.ones((num_samples, 1))

    # 3. Model outputs the true average velocity required to cross the whole trajectory
    average_velocity = model(x0, r, t)

    # 4. Single step integration: x_1 = x_0 + average_velocity * (t - r)
    # Since (t - r) = (1 - 0) = 1, it simplifies to:
    x1 = x0 + average_velocity

    return x1


In [22]:
# Sample from the trained model
generated_samples = mean_flow_sample(trained_mean_flow_model, num_samples=5)
print("\nGenerated 2D coordinates on the learned circle:")
print(generated_samples)

x shape: (5, 2), r shape: (5, 1), t shape: (5, 1)

Generated 2D coordinates on the learned circle:
[[-0.3838959   0.6198036 ]
 [ 0.5565535  -0.27457172]
 [-0.6684343  -0.01444781]
 [-3.7769747   1.3555549 ]
 [ 1.0909503   1.5317404 ]]
